# 1. Start

In [3]:
!nvidia-smi
#!pip list
!pip install sentencepiece

Wed Apr 22 19:23:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 590.48.01              Driver Version: 590.48.01      CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100 80GB PCIe          On  |   00000000:17:00.0 Off |                   On |
| N/A   54C    P0            102W /  300W |                  N/A   |     N/A      Default |
|                                         |                        |              Enabled |
+-----------------------------------------+-----

# 2. Configuration

In [5]:
# CHANGE BETWEEN RUNS
DATASET_SOURCE   = "alpaca"   # "alpaca" or "evol_instruct"
SELECTION_METHOD = "dalrs"    # "random", "length_only", or "dalrs"
BUDGET_M         = 1000       # 250, 500, 1000, 3000, 6000

# OOM SAFETY VALVE
MAX_SEQ_LENGTH   = 2048      

# FIXED FROM - train_mistral_7b.sh
MODEL_NAME         = "mistralai/Mistral-7B-v0.1"
WEIGHT_DECAY       = 0.1      # --weight_decay 0.1
LR_SCHEDULER       = "linear" # --lr_scheduler_type linear
ADAM_BETA1         = 0.9      # --adam_beta1 0.9
ADAM_BETA2         = 0.95     # --adam_beta2 0.95
NEFTUNE_ALPHA      = 3        # --neftune_noise_alpha 3
GRAD_CHECKPOINTING = True     # --gradient_checkpointing True
SAVE_STRATEGY      = "epoch"
SAVE_TOTAL_LIMIT   = 1
LOGGING_STEPS      = 1
PADDING_SIDE       = "right"  # from ModelArguments in train.py

# QLoRA ADAPTATIONS (hardware-forced, same for ALL conditions)
# Paper 1 used full SFT on 4×80GB A100. We use QLoRA on 1×9.5GB MIG slice.
LEARNING_RATE    = 2e-4       # Paper 1: 2e-6 -> 2e-4 for LoRA
BATCH_SIZE       = 1          # Paper 1: 2 -> 1 (VRAM)
GRAD_ACCUM       = 8          # Paper 1: 16 -> 8 (single GPU)
EPOCHS_MAP       = {250: 10, 500: 8, 1000: 6, 3000: 4, 6000: 3}
NUM_EPOCHS       = EPOCHS_MAP[BUDGET_M]

# Paper 2 DEITA Repr Filter — from combined_filter.py defaults
ALPHA            = 5          # candidate pool = ALPHA * BUDGET_M
REPR_FILTER_TAU  = 0.9        # cosine distance threshold
EMBED_BATCH_SIZE = 2          # safe for 9.5GB: ~3.5GB model + ~1.5GB activations

# HuggingFace token
HF_TOKEN = ""

import os
OUTPUT_DIR = f"./outputs/{DATASET_SOURCE}_{SELECTION_METHOD}_m{BUDGET_M}"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ["HF_HUB_DISABLE_XET"] = "1" # university cluster network friendly
os.environ["HF_TOKEN"] = HF_TOKEN

print(f"Experiment: {DATASET_SOURCE} | {SELECTION_METHOD} | m={BUDGET_M}")
print(f"Epochs: {NUM_EPOCHS} | Seq length: {MAX_SEQ_LENGTH} | Eff. batch: {BATCH_SIZE * GRAD_ACCUM}")
print(f"Output: {OUTPUT_DIR}")

Experiment: alpaca | dalrs | m=1000
Epochs: 6 | Seq length: 2048 | Eff. batch: 8
Output: ./outputs/alpaca_dalrs_m1000


# 3. Imports and Reproducibility

In [7]:
import json
import random
import pickle
import urllib.request
import numpy as np
import torch

from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
from trl import SFTTrainer

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    total  = props.total_memory / 1024**3
    used   = torch.cuda.memory_allocated() / 1024**3
    free   = total - used
    print(f"GPU:       {props.name}")
    print(f"Total:     {total:.2f} GB")
    print(f"Used:      {used:.2f} GB")
    print(f"Free:      {free:.2f} GB")
else:
    print("WARNING: No GPU detected.")

PyTorch: 2.7.1+cu128
GPU:       NVIDIA A100 80GB PCIe MIG 1g.10gb
Total:     9.50 GB
Used:      0.00 GB
Free:      9.50 GB


# Vicuna Prompt Template 
(Paper1 method)\
**Source:** `train.py` line 47 - `get_conversation_template('vicuna')`\
Paper 1 (Zhao et al., 2024 ICML) trains using FastChat's Vicuna template.\
Format: `USER: {instruction} ASSISTANT: {response}`  
`DataCollatorForCompletionOnlyLM` masks everything before `ASSISTANT:` from the loss, replicating `train.py`'s `IGNORE_TOKEN_ID` masking.\
Note: Paper 1's generation scripts use Alpaca format at inference — we match this in Cell 11.

In [4]:
VICUNA_SYSTEM = (
    "A chat between a curious user and an artificial intelligence assistant. "
    "The assistant gives helpful, detailed, and polite answers to the user's questions."
)
RESPONSE_TEMPLATE = "ASSISTANT:" # used later (Inference cell)

def format_vicuna(example):
    instruction = example["instruction"].strip()
    response    = example["output"].strip()
    user_turn   = f"{instruction}\n{example['input'].strip()}" if example.get("input", "").strip() else instruction
    return f"{VICUNA_SYSTEM}\n\nUSER: {user_turn} ASSISTANT: {response}"

print(format_vicuna({"instruction": "What is 2+2?", "input": "", "output": "4."}))

A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions.

USER: What is 2+2? ASSISTANT: 4.


# Load Data
For m=1000 with `length_only` or `dalrs`: downloads Paper 1's pre-filtered JSON directly from their GitHub.\
All other budgets/methods: loads full dataset from HuggingFace.

In [5]:
PAPER1_URLS = {
    "alpaca": (
        "https://raw.githubusercontent.com/tml-epfl/long-is-more-for-alignment/main/data/alpaca/filtered_alpaca_1k_longest.json"
    ),
    "evol_instruct": (
        "https://raw.githubusercontent.com/tml-epfl/long-is-more-for-alignment/main/data/evol-instruct/filtered_evol_instruct_1k_longest.json"
    ),
}


def download_paper1_json(source):
    # Download Paper 1's pre-filtered 1K JSON (FastChat format) from their GitHub.
    url = PAPER1_URLS[source]
    with urllib.request.urlopen(url) as r:
        data = json.loads(r.read().decode())
    print(f"Downloaded {len(data)} examples (FastChat format)")
    return data


def fastchat_to_dict(fastchat_data):
    # Parse FastChat format: example['conversations'] -> {instruction, input, output}.
    result = []
    for item in fastchat_data:
        convs = item.get("conversations", [])
        human = next((c["value"] for c in convs if c["from"] == "human"), "")
        gpt   = next((c["value"] for c in convs if c["from"] == "gpt"),   "")
        result.append({"instruction": human, "input": "", "output": gpt})
    return result


def load_full_dataset(source):
    if source == "alpaca":
        print("Loading Alpaca-52K...")
        ds = load_dataset("tatsu-lab/alpaca", split="train")
        return [{"instruction": r["instruction"], "input": r.get("input", ""), "output": r["output"]} for r in ds]
    elif source == "evol_instruct":
        print("Loading Evol-Instruct-70K...")
        ds = load_dataset("WizardLM/WizardLM_evol_instruct_70k", split="train")
        return [{"instruction": r["instruction"], "input": "", "output": r["output"]} for r in ds]


def length_prefilter(examples, budget_m, alpha):
    # Paper 1 Section 3.1: sort by response length, return top alpha*m candidates.
    # Forms candidate pool for DALRS stage 1
    pool = sorted(examples, key=lambda x: len(x["output"].split()), reverse=True)[:alpha * budget_m]
    lengths = [len(x["output"].split()) for x in pool]
    print(f"Length pre-filter: {len(pool)} candidates | range: {min(lengths)}–{max(lengths)} words")
    return pool


# Load data
if BUDGET_M == 1000 and SELECTION_METHOD in ["length_only", "dalrs"]:
    raw = download_paper1_json(DATASET_SOURCE)
    paper1_1k = fastchat_to_dict(raw)
    if SELECTION_METHOD == "length_only":
        train_examples = paper1_1k
        candidate_pool = paper1_1k
        print("Using Paper 1's exact pre-filtered 1K.")
    else:
        full = load_full_dataset(DATASET_SOURCE)
        candidate_pool = length_prefilter(full, BUDGET_M, ALPHA)
elif SELECTION_METHOD == "random":
    full = load_full_dataset(DATASET_SOURCE)
    candidate_pool = full
else:
    full = load_full_dataset(DATASET_SOURCE)
    candidate_pool = length_prefilter(full, BUDGET_M, ALPHA)

print(f"Candidate pool: {len(candidate_pool)} examples")

Downloaded 1000 examples (FastChat format)
Loading Alpaca-52K...
Length pre-filter: 5000 candidates | range: 100–717 words
Candidate pool: 5000 examples


# Data Selection (DALRS Pipeline) us
This cell is the core of the paper.\
\
**random** - control baseline. No selection intelligence.\
**length_only** - Paper 1 (Zhao et al., 2024 ICML). Top-m longest responses.\
**dalrs** - our method (Johnny & Kristen, CS 652).\
Stage 1: Paper 1's length pre-filter (already done above - candidate_pool).\
Stage 2: Paper 2's Repr Filter (Liu et al., 2024 ICLR) - greedy cosine-distance diversity pruning.\
\
Embedding model: Mistral-7B-v0.1 (same model used for fine-tuning).\
Following DEITA Appendix C.1 recommendation: use the training model for embeddings\
since its representations are most relevant to what the model will actually learn.\
\
VRAM during embedding:\
Mistral-7B 4-bit: ~3.5GB | Activations batch=2: ~1.5GB | Total: ~5GB\
Model is deleted and VRAM fully cleared before reloading Mistral for fine-tuning.

In [7]:
# uncomment and run this ONCE to download Mistral-7B model weights (saved to ./cache)
# Then RESTART KERNEL
#!HF_HUB_DISABLE_XET=1 hf download mistralai/Mistral-7B-v0.1 --token YOUR_HF_TOKEN_HERE --max-workers 1

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


Fetching 14 files: 100%|█████████████████████| 14/14 [00:00<00:00, 14987.30it/s]
/home/jovyan/.cache/huggingface/hub/models--mistralai--Mistral-7B-v0.1/snapshots/27d67f1b5f57dc0953326b2601d68371d40ea8da


In [6]:
def compute_embeddings(examples, batch_size=2):
    # Last-token hidden state embedding - matches clm_embedder.py (hkust-nlp/deita) lines 40-47.
    # Uses Mistral-7B (same model as fine-tuning) per DEITA Appendix C.1 model-based recommendation.
    # seq_len = attention_mask.sum(1, keepdim=True)
    # last_hidden = outputs.hidden_states[-1][arange, seq_len - 1]
    # Results cached to disk - only runs once per candidate pool.
    cache_path = f"{OUTPUT_DIR}/embeddings_cache.pkl"
    if os.path.exists(cache_path):
        print(f"Loading cached embeddings from {cache_path}")
        with open(cache_path, "rb") as f:
            return pickle.load(f)

    print(f"Embedding {len(examples)} examples with {MODEL_NAME} (4-bit)...")

    # padding_side='right' required — clm_embedder.py checks this for last-token indexing
    tok = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN, padding_side="right")
    tok.pad_token = tok.eos_token

    emb_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        token=HF_TOKEN,
        device_map="auto",
        output_hidden_states=True,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
        ),
    ).eval()

    used = torch.cuda.memory_allocated() / 1024**3
    free = torch.cuda.get_device_properties(0).total_memory / 1024**3 - used
    print(f"VRAM after embedding model load - Used: {used:.2f} GB | Free: {free:.2f} GB")

    all_embs = []
    with torch.no_grad():
        for i in range(0, len(examples), batch_size):
            batch = examples[i:i+batch_size]
            # DEITA embeds the full instruction+response concatenation
            texts = [
                f"{ex['instruction']}\n{ex['input']}\n{ex['output']}" if ex.get("input", "").strip()
                else f"{ex['instruction']}\n{ex['output']}"
                for ex in batch
            ]
            enc = tok(texts, return_tensors="pt", padding=True,
                      truncation=True, max_length=512).to("cuda")
            out = emb_model(**enc)

            # Last-token hidden state — clm_embedder.py lines 40-47
            seq_len     = enc["attention_mask"].sum(1, keepdim=True)  # (B, 1)
            last_hidden = out.hidden_states[-1]                        # (B, T, H)
            B           = seq_len.size(0)
            emb         = last_hidden[torch.arange(B)[:, None], seq_len - 1].squeeze(1)
            all_embs.append(emb.cpu().float().numpy())

            if i % (batch_size * 100) == 0:
                print(f"  {min(i+batch_size, len(examples))}/{len(examples)} embedded")

    # Delete embedding model and free VRAM before reloading Mistral for fine-tuning
    del emb_model
    torch.cuda.empty_cache()
    import gc; gc.collect()
    used = torch.cuda.memory_allocated() / 1024**3
    free = torch.cuda.get_device_properties(0).total_memory / 1024**3 - used
    print(f"VRAM after freeing embedder - Used: {used:.2f} GB | Free: {free:.2f} GB")

    embeddings = np.vstack(all_embs)
    with open(cache_path, "wb") as f:
        pickle.dump(embeddings, f)
    print(f"Embeddings cached: {embeddings.shape} -> {cache_path}")
    return embeddings


def repr_filter(pool, embeddings, budget_m, tau=0.9):
    # DEITA greedy diversity filter - Algorithm 1 (Liu et al., 2024 ICLR).
    # Source: hkust-nlp/deita combined_filter.py - threshold=0.9, cosine distance.
    # Iterates pool in length order (longest first).
    # Adds example to S if cosine distance to nearest neighbor in S >= (1-tau).
    # tau=0.9 -> reject if >90% similar to anything already selected.
    # Runs entirely on CPU - no VRAM used.
    reject_below = 1.0 - tau
    sel_idx  = []
    sel_embs = []

    for i in range(len(pool)):
        if len(sel_idx) >= budget_m:
            break
        emb = embeddings[i]
        if not sel_embs:
            sel_idx.append(i)
            sel_embs.append(emb)
            continue
        sel_arr  = np.array(sel_embs)
        emb_n    = emb     / (np.linalg.norm(emb) + 1e-8)
        sel_n    = sel_arr / (np.linalg.norm(sel_arr, axis=1, keepdims=True) + 1e-8)
        min_dist = 1.0 - (sel_n @ emb_n).max()
        if min_dist >= reject_below:
            sel_idx.append(i)
            sel_embs.append(emb)
        if len(sel_idx) % 100 == 0 and len(sel_idx) > 0:
            print(f"  {len(sel_idx)}/{budget_m} selected (scanned {i+1}/{len(pool)})")

    selected = [pool[i] for i in sel_idx]
    print(f"Repr Filter done: {len(selected)} from {len(pool)} candidates")
    if len(selected) < budget_m:
        print(f"WARNING: Only {len(selected)} diverse examples found (needed {budget_m}).")
        print("Fix: lower REPR_FILTER_TAU to 0.85 or increase ALPHA in Configuration.")
    return selected


# Run selection based on method
if SELECTION_METHOD == "random":
    random.seed(SEED)
    train_examples = random.sample(candidate_pool, min(BUDGET_M, len(candidate_pool)))
    print(f"Random: {len(train_examples)} examples selected")

elif SELECTION_METHOD == "length_only":
    # Already set for m=1000, otherwise slice from sorted pool
    if not (BUDGET_M == 1000 and "train_examples" in dir()):
        train_examples = candidate_pool[:BUDGET_M]
    print(f"Length-only: {len(train_examples)} examples selected")

elif SELECTION_METHOD == "dalrs":
    # Stage 2: embed candidate pool then apply Repr Filter
    embeddings     = compute_embeddings(candidate_pool, batch_size=EMBED_BATCH_SIZE)
    train_examples = repr_filter(candidate_pool, embeddings, BUDGET_M, REPR_FILTER_TAU)

# Save selected data for reproducibility
with open(f"{OUTPUT_DIR}/selected_data.json", "w") as f:
    json.dump(train_examples, f, indent=2)
print(f"Training set: {len(train_examples)} examples saved to {OUTPUT_DIR}/selected_data.json")

Embedding 5000 examples with mistralai/Mistral-7B-v0.1 (4-bit)...


The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

VRAM after embedding model load — Used: 4.17 GB | Free: 5.33 GB
  2/5000 embedded
  202/5000 embedded
  402/5000 embedded
  602/5000 embedded
  802/5000 embedded
  1002/5000 embedded
  1202/5000 embedded
  1402/5000 embedded
  1602/5000 embedded
  1802/5000 embedded
  2002/5000 embedded
  2202/5000 embedded
  2402/5000 embedded
  2602/5000 embedded
  2802/5000 embedded
  3002/5000 embedded
  3202/5000 embedded
  3402/5000 embedded
  3602/5000 embedded
  3802/5000 embedded
  4002/5000 embedded
  4202/5000 embedded
  4402/5000 embedded
  4602/5000 embedded
  4802/5000 embedded
VRAM after freeing embedder — Used: 0.15 GB | Free: 9.35 GB
Embeddings cached: (5000, 4096) -> ./outputs/alpaca_dalrs_m1000/embeddings_cache.pkl
  100/1000 selected (scanned 102/5000)
  200/1000 selected (scanned 210/5000)
  300/1000 selected (scanned 318/5000)
  400/1000 selected (scanned 425/5000)
  500/1000 selected (scanned 530/5000)
  600/1000 selected (scanned 638/5000)
  700/1000 selected (scanned 750/5000)


# Format Dataset
Apply the Vicuna template to every selected training example.\
Converts to a HuggingFace Dataset object that SFTTrainer can consume.\
Prints the first example so you can visually confirm the format looks correct.

In [8]:
hf_dataset = Dataset.from_dict({"text": [format_vicuna(ex) for ex in train_examples]})
print(f"Dataset: {len(hf_dataset)} examples formatted with Vicuna template")
print("\nFirst example (first 300 chars):")
print(hf_dataset[0]["text"][:300])

Dataset: 1000 examples formatted with Vicuna template

First example (first 300 chars):
A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions.

USER: Explain a common misconception about your topic.
Topic: Using AI to Augment Human Capabilities ASSISTANT: A common misconception about AI 


# Load Mistral-7B in 4bit NF4
Paper 1 used full bf16 weights (14GB). We use 4-bit NF4 (3.5GB) to fit 9.5GB.\
bf16 compute dtype matches Paper 1's `--bf16 True` at the compute level.\
`use_cache=False` matches train.py which disables this before training.\
If headroom after load is < 4GB -> set MAX_SEQ_LENGTH=1024 in Configuration.

In [9]:
# Check VRAM is clear — embedding model should already be freed from Data Selection cell
total = torch.cuda.get_device_properties(0).total_memory / 1024**3
used  = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM before load — Used: {used:.2f} GB | Free: {total-used:.2f} GB")
if used > 1.0:
    print("WARNING: >1GB still allocated — restart kernel to fully clear VRAM.")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,  # --bf16 True from train_mistral_7b.sh
    bnb_4bit_use_double_quant=True,          # saves ~0.4GB extra
)

print(f"Loading {MODEL_NAME} in 4-bit NF4...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
    use_cache=False,  # train.py sets config.use_cache = False before training
)

# Tokenizer settings from train.py ModelArguments:
# padding_side='right', use_fast=False, model_max_length from --model_max_length
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    token=HF_TOKEN,
    model_max_length=MAX_SEQ_LENGTH,
    padding_side=PADDING_SIDE,
    use_fast=False,
    trust_remote_code=True,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # Mistral has no unk_token

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=GRAD_CHECKPOINTING)

used  = torch.cuda.memory_allocated() / 1024**3
free  = total - used
print(f"VRAM after load — Used: {used:.2f} GB | Free: {free:.2f} GB")
if free < 4.0:
    print("WARNING: <4GB headroom — set MAX_SEQ_LENGTH=1024 in Configuration.")

VRAM before load — Used: 0.01 GB | Free: 9.49 GB
Loading mistralai/Mistral-7B-v0.1 in 4-bit NF4...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

VRAM after load — Used: 4.34 GB | Free: 5.16 GB


# LoRA Adapters
Attach LoRA adapters to all attention and MLP projection layers.\
Only adapter weights update during training (~0.3GB, ~0.6% of parameters).\
Base model stays frozen at 4-bit — this is what makes QLoRA memory efficient.

In [10]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    # All attention + MLP projection layers
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,   # Paper 1 uses no dropout
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

used = torch.cuda.memory_allocated() / 1024**3
free = total - used
print(f"VRAM after LoRA - Used: {used:.2f} GB | Free: {free:.2f} GB")

trainable params: 41,943,040 || all params: 7,283,675,136 || trainable%: 0.5758
VRAM after LoRA — Used: 4.50 GB | Free: 5.00 GB


# Train
SFTTrainer setup follows exam notebook pattern (trl 0.21.0 compatible): 

- `processing_class=tokenizer` replaces deprecated `tokenizer=` argument
- Pre-tokenized dataset passed directly — no internal re-tokenization
- `TrainingArguments` passed inline inside `SFTTrainer` (cleaner for trl 0.21.0)
- `DataCollatorForLanguageModeling` (mlm=False) replaces removed `DataCollatorForCompletionOnlyLM`\
NEFTune alpha=3 matches `train_mistral_7b.sh` exactly.\
`use_cache=True` is restored after training — matches train.py behavior.\
Saves LoRA adapter weights on completion (~10-100MB per condition).

In [ ]:
import time

# Pre-tokenize dataset - same approach as exam notebook cell 4
# Avoids SFTTrainer re-tokenizing internally, which is cleaner with trl 0.21.0
dataset_tokenized = hf_dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, max_length=MAX_SEQ_LENGTH),
    batched=True,
    num_proc=1,
    remove_columns=hf_dataset.column_names,
)
print(f"Tokenized dataset: {len(dataset_tokenized)} examples")

# DataCollatorForLanguageModeling replaces DataCollatorForCompletionOnlyLM (removed trl 0.21.0)
# mlm=False -> causal LM (predict next token), not masked LM (BERT-style)
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

# SFTTrainer setup - modeled after exam notebook cell 4
# processing_class= replaces deprecated tokenizer= (trl 0.21.0)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset_tokenized,
    data_collator=collator,
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        # From train_mistral_7b.sh exactly
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,               # 0.1 
        lr_scheduler_type=LR_SCHEDULER,         # linear 
        adam_beta1=ADAM_BETA1,                   # 0.9 
        adam_beta2=ADAM_BETA2,                   # 0.95 
        gradient_checkpointing=GRAD_CHECKPOINTING,
        save_strategy=SAVE_STRATEGY,
        save_total_limit=SAVE_TOTAL_LIMIT,
        logging_steps=LOGGING_STEPS,
        eval_strategy="no",
        bf16=True, fp16=False,
        # QLoRA adaptations (hardware-forced)
        learning_rate=LEARNING_RATE,             # ADAPTED: 2e-6 -> 2e-4 for LoRA
        per_device_train_batch_size=BATCH_SIZE,  # ADAPTED: 2 -> 1 (VRAM)
        gradient_accumulation_steps=GRAD_ACCUM, # ADAPTED: 16 -> 8 (single GPU)
        optim="paged_adamw_8bit",               # ADAPTED: 8-bit paged optimizer
        dataloader_num_workers=0,               # saves ~0.5GB vs multi-worker
        dataloader_pin_memory=False,
        neftune_noise_alpha=NEFTUNE_ALPHA,   # 3 from train_mistral_7b.sh
        report_to="none",
        seed=SEED,
    ),
)

used = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM before training - Used: {used:.2f} GB | Free: {total-used:.2f} GB")
if used > total * 0.85:
    print("WARNING: >85% VRAM — set MAX_SEQ_LENGTH=1024 in Configuration and restart.")

print("Starting training...")
t0 = time.time()
trainer.train()
train_time = time.time() - t0
peak_vram  = torch.cuda.max_memory_allocated() / 1024**3
print(f"Training complete! Time: {train_time/60:.1f} min | Peak VRAM: {peak_vram:.2f} GB")

trainer.model.config.use_cache = True  # restore - matches train.py
adapter_path = f"{OUTPUT_DIR}/final_adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Adapter saved: {adapter_path}")

# Save weights from memory to json

In [1]:
import json

# Load loss history from the saved checkpoint — works even after kernel restart
checkpoint_path = f"{OUTPUT_DIR}/checkpoint-{int(NUM_EPOCHS * len(train_examples) / (BATCH_SIZE * GRAD_ACCUM))}/trainer_state.json"
with open(checkpoint_path) as f:
    state = json.load(f)

loss_data = {
    "experiment": f"{DATASET_SOURCE}_{SELECTION_METHOD}_m{BUDGET_M}",
    "steps":  [x["step"]  for x in state["log_history"] if "loss" in x],
    "losses": [x["loss"]  for x in state["log_history"] if "loss" in x],
    "epochs": [x["epoch"] for x in state["log_history"] if "loss" in x],
}

loss_path = f"{OUTPUT_DIR}/loss_curve_{DATASET_SOURCE}_{SELECTION_METHOD}_m{BUDGET_M}.json"
with open(loss_path, "w") as f:
    json.dump(loss_data, f, indent=2)

print(f"Steps: {len(loss_data['steps'])} | Final loss: {loss_data['losses'][-1]:.4f}")
print(f"Saved: {loss_path}")

Steps: 750 | Final loss: 1.7880


---

**SKIP IF MODEL IS STILL IN MEMORY**\
Run this only if the kernel restarted after training completed.\
Loads the saved adapter from disk instead of retraining.\
Configuration matches what was used during training

In [8]:
MODEL_NAME   = "mistralai/Mistral-7B-v0.1"
adapter_path = "./outputs/alpaca_dalrs_m1000/final_adapter"
HF_TOKEN     = "YOUR_HF_TOKEN_HERE"

DATASET_SOURCE   = "alpaca"
SELECTION_METHOD = "dalrs"
BUDGET_M         = 1000

# Load base model in 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=True,
)

# Load the LoRA adapter on top
model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

# Load tokenizer from saved adapter
tokenizer = AutoTokenizer.from_pretrained(adapter_path)

used = torch.cuda.memory_allocated() / 1024**3
print(f"Model loaded — VRAM: {used:.2f} GB")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded — VRAM: 4.00 GB


---

# Sanity Check Inference
Uses Alpaca format - matches Paper 1's `generate_alpacaeval.py` and `generate_test.py`.\
Note: training used Vicuna format. This train/inference mismatch exists in Paper 1's own repo.\
Test instructions from Paper 1 Figure 8 - compare your outputs to theirs qualitatively.\
`max_new_tokens=*` keeps KV cache small and prevents OOM on 9.5GB.

In [13]:
def generate(instruction, input_text="", max_new_tokens=2048):# 256
    # Alpaca inference format — matches generate_alpacaeval.py and generate_test.py exactly.
    # max_new_tokens caps KV cache growth safely within 9.5GB.
    if input_text.strip():
        prompt = (
            "Below is an instruction that describes a task, paired with an input "
            "that provides further context. "
            "Write a response that appropriately completes the request.\r\n\r\n"
            f"### Instruction:\r\n{instruction}\r\n\r\n"
            f"### Input:\r\n{input_text}\r\n\r\n### Response:"
        )
    else:
        prompt = (
            "Below is an instruction that describes a task. "
            "Write a response that appropriately completes the request.\r\n\r\n"
            f"### Instruction:\r\n{instruction}\r\n\r\n### Response:"
        )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,  # greedy — deterministic sanity check
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0][len(prompt):].strip()


model.eval()
print(f"=== Mistral-7B | {SELECTION_METHOD.upper()} | m={BUDGET_M} | {DATASET_SOURCE} ===")
# Test instructions from Paper 1 Figure 8 case study
for inst in [
    "Give me a sample 5 day itinerary for a Switzerland holiday, starting from Basel.",
    "As a pirate captain, what would you say to your crew to motivate them to search for hidden treasure?",
    "What language does Argentina people speak?",
]:
    print(f"\nQ: {inst}\nA: {generate(inst, max_new_tokens=2048)}\n" + "-" * 60)

=== Mistral-7B | DALRS | m=1000 | alpaca ===

Q: Give me a sample 5 day itinerary for a Switzerland holiday, starting from Basel.
A: > Dear Traveler,
>
> Thank you for your inquiry about a 5-day itinerary for a Switzerland holiday, starting from Basel. Switzerland is a beautiful country with a lot to offer, and I am happy to help you plan your trip.
>
> Here is a sample itinerary that I have put together for you:
>
> Day 1: Arrive in Basel and explore the city. Visit the Basel Zoo, the Kunstmuseum, and the Basel Paper Mill Museum.
>
> Day 2: Take a train to Lucerne and explore the city. Visit the Chapel Bridge, the Lion Monument, and the Glacier Garden.
>
> Day 3: Take a train to Interlaken and explore the city. Visit the Harder Kulm, the Jungfraujoch, and the Schilthorn.
>
> Day 4: Take a train to Zermatt and explore the city. Visit the Matterhorn, the Gornergrat, and the Klein Matterhorn.
>
> Day 5: Take a train back to Basel and explore the city. Visit the Basel Zoo, the Kunstmuseum

# Evaluation Commands
Run these after all 30 conditions are trained.\
lm-eval supports PEFT adapters natively via `pretrained=adapter_path,load_in_4bit=True`.\
Note: `data/test/` is the correct path for generate_test.py - not the script's hardcoded default.

In [ ]:
print(f"""
=== Run Complete ===
Adapter: {OUTPUT_DIR}/final_adapter
Data:    {OUTPUT_DIR}/selected_data.json

=== Next runs — change in Configuration cell ===
SELECTION_METHOD in [random, length_only, dalrs]
BUDGET_M         in [250, 500, 1000, 3000, 6000]
DATASET_SOURCE   in [alpaca, evol_instruct]
OOM fix:          set MAX_SEQ_LENGTH=1024

=== Evaluation (Paper 1 scripts) ===

1. AlpacaEval 2.0 (free):
   python generation/generate_alpacaeval.py \\
     --model_name_or_path {OUTPUT_DIR}/final_adapter \\
     --save_path results/alpacaeval/ \\
     --save_name mistral_{SELECTION_METHOD}_m{BUDGET_M}
   alpaca_eval evaluate --model_outputs results/alpacaeval/mistral_{SELECTION_METHOD}_m{BUDGET_M}_outputs.json

2. Test sets (correct path is data/test/ — not the script default):
   python generation/generate_test.py \\
     --model_name_or_path {OUTPUT_DIR}/final_adapter \\
     --test_path data/test/ --save_path results/test_responses/

3. Open LLM Leaderboard (free, no API key):
   lm_eval --model hf \\
     --model_args pretrained={OUTPUT_DIR}/final_adapter,load_in_4bit=True \\
     --tasks arc_challenge,mmlu,truthfulqa_mc2,winogrande \\
     --device cuda:0 --batch_size 1

4. MT-Bench (requires OpenAI API key):
   python gen_model_answer.py --model-path {OUTPUT_DIR}/final_adapter --model-id mistral_{SELECTION_METHOD}_m{BUDGET_M}
   python gen_judgment.py --model-list mistral_{SELECTION_METHOD}_m{BUDGET_M}
   python show_result.py  --model-list mistral_{SELECTION_METHOD}_m{BUDGET_M}
""")